In [77]:
import pandas as pd

# Load the original intent dataset
intent_df = pd.read_csv("amazon_intent_base.csv")

# Random 1,000 queries
sample_1000 = (
    intent_df[
        ["customer_tweet_id", "clean_customer_query"]
    ]
    .dropna(subset=["clean_customer_query"])
    .sample(n=1000, random_state=42)
    .reset_index(drop=True)
)

print("Total queries selected:", len(sample_1000))
print("\nFirst 10 queries:")
print(sample_1000.head(10).to_string(index=False))

Total queries selected: 1000

First 10 queries:
 customer_tweet_id                                                                                                                                                                                          clean_customer_query
          594532.0                                                                                                                                                                                                           UPS
          588782.0 I just got a package (a birthday gift for tomorrow) and the packaging and the item itself smell very strongly of smoke. I'm assuming the driver at Intelcom was smoking in the car and now my gift is ruined.
         1704079.0                                                                                             Once again, I'm not getting my packages on time from #Intelcom. So much for 2 day delivery! This is the 5th time!
          831799.0                                  

In [81]:
import json
import os
import gc
import time
import pandas as pd

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

BATCH_SIZE = 10

INPUT_FILE = "amazon_intent_base.csv"
SAMPLE_FILE = "amazon_sample_1000.csv"
RESULT_FILE = "amazon_discovery_results.jsonl"
TAXONOMY_FILE = "amazon_candidate_taxonomy.json"

# --------------------------------------------------
# LOAD ONLY WHAT WE NEED
# --------------------------------------------------

intent_df = pd.read_csv(
    INPUT_FILE,
    usecols=["customer_tweet_id", "clean_customer_query"]
)

# Create the 1000-query sample only once
if os.path.exists(SAMPLE_FILE):

    sample_1000 = pd.read_csv(SAMPLE_FILE)

    print("Loaded existing 1000-query sample.")

else:

    sample_1000 = (
        intent_df[
            ["customer_tweet_id", "clean_customer_query"]
        ]
        .dropna(subset=["clean_customer_query"])
        .sample(n=1000, random_state=42)
        .reset_index(drop=True)
    )

    sample_1000.to_csv(
        SAMPLE_FILE,
        index=False
    )

    print("Created and saved 1000-query sample.")

# We no longer need the full dataset in memory
del intent_df
gc.collect()

print("Queries:", len(sample_1000))


# --------------------------------------------------
# LOAD OR INITIALIZE TAXONOMY
# --------------------------------------------------

if os.path.exists(TAXONOMY_FILE):

    with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
        discovered_intents = json.load(f)

    print("Loaded existing taxonomy.")

else:

    discovered_intents = [
        {
            "name": "Non-Informative",
            "definition": (
                "A message containing no meaningful customer-support "
                "request, problem, or actionable information."
            )
        }
    ]

    with open(TAXONOMY_FILE, "w", encoding="utf-8") as f:
        json.dump(
            discovered_intents,
            f,
            indent=2,
            ensure_ascii=False
        )


# --------------------------------------------------
# FIND ALREADY PROCESSED BATCHES
# --------------------------------------------------

processed_ids = set()

if os.path.exists(RESULT_FILE):

    with open(RESULT_FILE, "r", encoding="utf-8") as f:

        for line in f:

            try:
                record = json.loads(line)

                processed_ids.add(
                    str(record["customer_tweet_id"])
                )

            except:
                pass

print("Already processed:", len(processed_ids))


# --------------------------------------------------
# DISCOVERY PROMPT
# --------------------------------------------------

def build_discovery_prompt(batch, current_intents):

    intent_text = "\n".join(
        f"- {x['name']}: {x['definition']}"
        for x in current_intents
    )

    query_text = "\n".join(
        f"{i+1}. ID={row.customer_tweet_id} | "
        f"{row.clean_customer_query}"
        for i, row in enumerate(
            batch.itertuples(index=False)
        )
    )

    return f"""
You are designing an intent taxonomy for Amazon customer support.

CURRENT INTENTS:

{intent_text}

CUSTOMER QUERIES:

{query_text}

Assign exactly ONE intent to every query.

RULES:

1. Reuse an existing intent ONLY when it genuinely represents
   the same underlying customer problem.

2. NEVER force a meaningful customer problem into an unrelated
   existing intent.

3. If no existing intent genuinely fits, create a new meaningful
   intent.

4. New intents should represent customer-support problems, not
   individual products, names, countries, or wording variations.

5. Keep intents reasonably broad.

6. Non-Informative is ONLY for messages containing no meaningful
   support request, problem, or actionable information.

7. Do NOT merge, rename, or delete existing intents.

8. Return exactly one label for every query.

9. Only include genuinely NEW intents in new_intents.

Return ONLY valid JSON:

{{
  "new_intents": [
    {{
      "name": "Intent Name",
      "definition": "Short definition"
    }}
  ],
  "labels": [
    {{
      "id": "customer tweet id",
      "intent": "Intent Name"
    }}
  ]
}}
"""


# --------------------------------------------------
# PROCESS BATCHES
# --------------------------------------------------

total = len(sample_1000)

for start in range(0, total, BATCH_SIZE):

    batch = sample_1000.iloc[
        start:start + BATCH_SIZE
    ].copy()

    # Skip queries already processed
    batch = batch[
        ~batch["customer_tweet_id"]
        .astype(str)
        .isin(processed_ids)
    ]

    if len(batch) == 0:
        continue

    print(
        f"\nProcessing queries "
        f"{start + 1}-{min(start + BATCH_SIZE, total)}..."
    )

    prompt = build_discovery_prompt(
        batch,
        discovered_intents
    )

    try:

        result = ask_ollama_json(prompt)

        # ------------------------------------------
        # ADD NEW INTENTS
        # ------------------------------------------

        existing_names = {
            x["name"].lower()
            for x in discovered_intents
        }

        for new_intent in result.get(
            "new_intents",
            []
        ):

            name = new_intent["name"].strip()

            if name.lower() not in existing_names:

                discovered_intents.append({
                    "name": name,
                    "definition": (
                        new_intent["definition"]
                        .strip()
                    )
                })

                existing_names.add(
                    name.lower()
                )

        # ------------------------------------------
        # WRITE RESULTS IMMEDIATELY TO DISK
        # ------------------------------------------

        with open(
            RESULT_FILE,
            "a",
            encoding="utf-8"
        ) as f:

            for label in result.get(
                "labels",
                []
            ):

                record = {
                    "customer_tweet_id":
                        str(label["id"]),

                    "intent":
                        label["intent"].strip()
                }

                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                processed_ids.add(
                    str(label["id"])
                )

        # ------------------------------------------
        # SAVE TAXONOMY
        # ------------------------------------------

        with open(
            TAXONOMY_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                discovered_intents,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            "Processed:",
            len(processed_ids),
            "| Intents:",
            len(discovered_intents)
        )

    except Exception as e:

        print(
            "Batch failed:",
            start,
            "-",
            start + BATCH_SIZE,
            "|",
            repr(e)
        )

    # ------------------------------------------
    # RELEASE MEMORY
    # ------------------------------------------

    del batch
    del prompt

    if "result" in locals():
        del result

    gc.collect()

    # Small pause so the system can recover
    time.sleep(0.5)


print("\n==============================")
print("DISCOVERY FINISHED")
print("==============================")
print("Queries processed:", len(processed_ids))
print("Candidate intents:", len(discovered_intents))
print("Results saved to:", RESULT_FILE)
print("Taxonomy saved to:", TAXONOMY_FILE)

Created and saved 1000-query sample.
Queries: 1000
Already processed: 0

Processing queries 1-10...


KeyboardInterrupt: 

In [84]:
import gc

# Release objects from the interrupted run
gc.collect()

print("Cleaned Python garbage.")

Cleaned Python garbage.


In [85]:
!sync && echo 1 | sudo tee /proc/sys/vm/drop_caches


tee: /proc/sys/vm/drop_caches: Read-only file system
1
